# Section 2 — Euler's Method

[← Direction Fields](01_direction_fields_autonomous_odes.ipynb) ·
[Module 3](README.md) ·
[Next: Euler Error →](03_euler_error.ipynb)

For the IVP

$$
y'=f(t,y),\qquad y(t_0)=y_0,
$$

Euler's method uses the tangent-line update

$$
w_{n+1}=w_n+h f(t_n,w_n),
\qquad
t_{n+1}=t_n+h.
$$

### Learning goals

- derive Euler's method from a tangent-line approximation;
- perform Euler steps by hand;
- implement Euler's method in Python;
- compare numerical and exact solutions;
- connect discrete Euler data back to interpolation from Module 2.


In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np


## 1. Course example

Consider

$$
y'=t e^{3t}-2y,
\qquad
y(0)=0,
$$

on $0\le t\le1$ with $h=0.5$.


In [2]:
def f(t, y):
    """Right-hand side of the course Euler example."""
    return t * math.exp(3.0 * t) - 2.0 * y


def euler(function, t0, y0, h, n_steps):
    """Return Euler nodes and approximations."""
    t_values = np.empty(n_steps + 1)
    w_values = np.empty(n_steps + 1)

    t_values[0] = t0
    w_values[0] = y0

    for n in range(n_steps):
        w_values[n + 1] = (
            w_values[n]
            + h * function(
                t_values[n],
                w_values[n],
            )
        )
        t_values[n + 1] = t_values[n] + h

    return t_values, w_values


t_values, w_values = euler(
    f,
    0.0,
    0.0,
    0.5,
    2,
)

for t_value, w_value in zip(
    t_values,
    w_values,
):
    print(
        f"t = {t_value:.1f}, "
        f"w = {w_value:.8f}"
    )


t = 0.0, w = 0.00000000
t = 0.5, w = 0.00000000
t = 1.0, w = 1.12042227


## 2. Exact solution comparison

For this problem,

$$
y(t)
=
e^{3t}
\left(
\frac{t}{5}
-\frac1{25}
\right)
+
\frac1{25}e^{-2t}.
$$


In [3]:
def exact(t):
    return (
        np.exp(3.0 * t)
        * (t / 5.0 - 1.0 / 25.0)
        + np.exp(-2.0 * t) / 25.0
    )


t_fine = np.linspace(0.0, 1.0, 400)

plt.figure(figsize=(9, 5))
plt.plot(
    t_fine,
    exact(t_fine),
    label="exact solution",
)
plt.plot(
    t_values,
    w_values,
    "o-",
    label="Euler, h=0.5",
)
plt.xlabel("t")
plt.ylabel("y")
plt.title("Euler's Method")
plt.grid(alpha=0.25)
plt.legend()
plt.show()

print(
    "error at t=1 = "
    f"{abs(exact(1.0) - w_values[-1]):.8f}"
)


error at t=1 = 2.09867705


## 3. Euler data can be interpolated

The Exam 3 review deliberately connects Modules 2 and 3.

Consider

$$
y'=e^{x-y},
\qquad
y(0)=1,
$$

whose exact solution is

$$
y(x)=\ln(e^x+e-1).
$$

First use Euler with $h=0.2$ to generate the data

$$
(0,w_0),(0.2,w_1),\ldots,(1,w_5).
$$

Then construct the degree-at-most-five Lagrange polynomial through those
**numerical** data points and use it to approximate $y(0.7)$.

This separates two errors:

1. Euler already introduces error into the node values;
2. interpolation then approximates between those numerical nodes.


In [4]:
def review_rhs(x, y):
    return math.exp(x - y)


def review_exact(x):
    return np.log(
        np.exp(x)
        + math.e
        - 1.0
    )


x_nodes, euler_nodes = euler(
    review_rhs,
    0.0,
    1.0,
    0.2,
    5,
)

print("Euler-generated data")
for x_value, y_value in zip(
    x_nodes,
    euler_nodes,
):
    print(
        f"x = {x_value:.1f}, "
        f"w = {y_value:.10f}"
    )


Euler-generated data
x = 0.0, w = 1.0000000000
x = 0.2, w = 1.0735758882
x = 0.4, w = 1.1570671084
x = 0.6, w = 1.2508751194
x = 0.8, w = 1.3551929443
x = 1.0, w = 1.4699852459


In [5]:
def lagrange_value(
    x_nodes,
    y_nodes,
    x_value,
):
    """Evaluate a Lagrange interpolant at one point."""
    total = 0.0

    for i, x_i in enumerate(x_nodes):
        basis = 1.0

        for j, x_j in enumerate(x_nodes):
            if i != j:
                basis *= (
                    (x_value - x_j)
                    / (x_i - x_j)
                )

        total += y_nodes[i] * basis

    return total


x_value = 0.7
interpolated = lagrange_value(
    x_nodes,
    euler_nodes,
    x_value,
)
exact_at_point = review_exact(x_value)

print(
    f"P5(0.7)       = {interpolated:.10f}"
)
print(
    f"exact y(0.7)  = {exact_at_point:.10f}"
)
print(
    "absolute error = "
    f"{abs(interpolated - exact_at_point):.3e}"
)


P5(0.7)       = 1.3017169280
exact y(0.7)  = 1.3169535369
absolute error = 1.524e-02


### References

- Leon Q. Brin, *Tea Time Numerical Analysis*.
- Richard L. Burden and J. Douglas Faires, *Numerical Analysis*.
- MAD 4401 Euler lecture and Exam 3 review.

[Next: Euler Error →](03_euler_error.ipynb)
